# 5. Modelo autorregresivo lineal

El valor de cada mes se explica con los meses anteriores; el numero de rezagos se elige por validacion. El tramo de prueba se pronostica recursivamente.

In [1]:
%run functions.ipynb

In [2]:
from sklearn.linear_model import LinearRegression

df = load_data()

# Seleccion del numero de rezagos con una validacion dentro del
# entrenamiento: se reservan los ultimos 24 meses del entrenamiento.
train = df.yt_true.iloc[:-24]

validacion = {}

for p_max in (12, 13, 24, 36):
    forecast = recursive_forecast(LinearRegression(), train, p_max)
    validacion[p_max] = np.mean(np.abs(train.values[-24:] - forecast[-24:]))

pd.Series(validacion, name="MAE validacion").round(1)

12    564.7
13    507.0
24    501.1
36    385.4
Name: MAE validacion, dtype: float64

In [3]:
# Se usa el numero de rezagos con menor error de validacion; el tramo
# de prueba no interviene en la eleccion.

P_MAX = min(validacion, key=validacion.get)
print("P_MAX =", P_MAX)

df["yt_pred_ar"] = recursive_forecast(LinearRegression(), df.yt_true, P_MAX)

plot_time_series(df[["yt_true", "yt_pred_ar"]])

P_MAX = 36


<Figure size 1200x400 with 1 Axes>

In [4]:
metrics = compute_evaluation_metrics(df)
save_forecasts(df)
save_metrics(metrics)
metrics

,Metrics,yt_pred_ar
0,MSE Train,50533.08
1,MSE Test,92327.44
2,MAE Train,160.48
3,MAE Test,224.08


Al usar los rezagos, el modelo se adapta al nivel reciente de la serie y el error en prueba cae a menos de la mitad del modelo de tendencia + mes.